Load libraries and functions

In [1]:
from poisson_bcann_vi import *
from codebase_0 import CANN, MLP, cann_training_loop, mlp_train_loop, DeviceLoader
import pandas as pd
from sklearn.model_selection import train_test_split
from poisson_bcann_hmc_nuts import (
    stratified_subsample, make_chain_inits, run_nuts, time_nuts, get_device, 
)
from sklearn.preprocessing import MinMaxScaler
import numpy as np
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from poisson_nuts_parallel import run_nuts_parallel

seed = 42
set_seed(seed)
pyro.clear_param_store()

/workspace/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load data and perform train/validation/test split

In [2]:
device = torch.device("cuda:0")

dataset = pd.read_csv("freMTPL2freq_transformed.csv", index_col= 0) 
y = torch.tensor(dataset["ClaimNb"].to_numpy(), dtype=torch.float32)
exposure = torch.tensor(dataset["Exposure"].to_numpy(), dtype=torch.float32)
X = torch.tensor(dataset.drop(columns=["IDpol", "Exposure","ClaimNb"]).to_numpy(), dtype=torch.float32)   # 48 columns

# Learning-test set split
X_learn, X_test, y_learn, y_test, exp_learn, exp_test = train_test_split(
    X, y, exposure, test_size= 0.1, random_state= seed
)

# Training-CV set split
X_train, X_val, y_train, y_val, exp_train, exp_val = train_test_split(
    X_learn, y_learn, exp_learn, test_size= 0.1, random_state= seed
)

learn_scaler = MinMaxScaler().fit(X_learn)
X_learn = learn_scaler.transform(X_learn)
X_test = learn_scaler.transform(X_test)

train_scaler = MinMaxScaler().fit(X_train)
X_train = train_scaler.transform(X_train)
X_val = train_scaler.transform(X_val)

def T(a):
    return torch.as_tensor(a, dtype=torch.float32, device=device)

X_train, X_val, X_learn, X_test = map(T, (X_train, X_val, X_learn, X_test))
y_train, y_val, y_learn, y_test = map(T, (y_train, y_val, y_learn, y_test))
exp_train, exp_val, exp_learn, exp_test = map(T, (exp_train, exp_val, exp_learn, exp_test))

torch.set_default_device(device)
train_loader = DeviceLoader(
        X_train, y_train, torch.log(exp_train),
        batch_size = 10000,
        shuffle= True
)
val_loader = DeviceLoader(
        X_val, y_val, torch.log(exp_val),
        batch_size= 10000
)
learn_loader = DeviceLoader(
        X_learn, y_learn, torch.log(exp_learn),
        batch_size= 10000
)

Initialise PoissonBCANN and VI posterior (as guide) 

In [3]:
network_architecture = [20,15, 10]

model = PoissonBCANN(
    in_features= 48,
    neural_net_params={"hidden_activation_nodes": network_architecture,
                        "hidden_activations": nn.Tanh,
                        "prior_scale": 1.0},
).to(device)
glm_mle = poisson_fit_glm_mle(
    X_train, y_train, 
    log_exposure=torch.log(exp_train), 
    random_state=seed
)
guide = build_bcann_guide(
    model, 
    X_train[:256], 
    y_train[:256], 
    glm_mle=glm_mle, 
    rank=10
)

bcann_history = poisson_bcann_svi_training_loop(
    model, guide, learn_loader, val_loader,
    n_epochs=55, 
    lr=5e-3, 
    val_every=100, # set this to only perform training and no early stopping CV checks
    patience=10, 
    random_state=seed, 
    device = device
)


Epoch 1 | -ELBO/n: 0.2132 | val deviance: 0.3201


Create a chain object for NUTS, run a test run to see how long it takes per iteration, and run NUTS.

In [5]:
inits = make_chain_inits(
    model, X_learn, y_learn, torch.log(exp_learn), num_chains=6, glm_mle=glm_mle,
    guide=guide, how="svi_sample", seed=seed, device = device
)

time_nuts(model, X_learn, y_learn, torch.log(exp_learn), inits, jit_compile= True)

poisson_hmc_samples, poisson_hmc_info = run_nuts_parallel(
    dict(
        in_features=48, 
        neural_net_params={
            "hidden_activation_nodes": network_architecture,
            "hidden_activations": nn.Tanh
        }
    ), 
    X_learn, 
    y_learn, 
    torch.log(exp_learn), 
    inits, 
    model_cls = PoissonBCANN,
    out_dir = "poisson_nuts_chains",
    num_warmup=1000, 
    num_samples=1000,
    target_accept=0.8, 
    max_tree_depth=8, 
    seed=seed, 
    device = device,
    progress_every = 5,
    jit_compile = True
)

elapsed 2:27:26
chain 5: Sample: 100%|██████████| 2000/2000 [2:26:29,  4.39s/it, step size=3.63e-02, acc. prob=0.945]


Write to disk for later uses

In [7]:
torch.save(
    {
        "samples": poisson_hmc_samples,
        "info": poisson_hmc_info,
        "config": dict(num_warmup=1000, num_samples=1000, target_accept=0.8,
                       max_tree_depth=8, seed=seed, n_chains=6),
    },
    "hmc_results/poisson_hmcnuts_full_set.pt",
)